# 입력 조건 커버리지 테스트

폼이 받는 입력 12개가 **합성 고객에 전부 들어 있는지**, 그리고 그 고객을 다시
폼에 넣었을 때 **추천이 규칙을 지키는지**를 왕복으로 확인한다.

```
합성 고객 한 명  ->  build_query()  ->  recommend()  ->  위반 검사
```

합성 데이터가 없거나 낡았으면 먼저 `python src/make_synthetic.py`.

In [ ]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd() / "src"))

from make_synthetic import OUT_PATH as SYNTHETIC_PATH
from profile_input import InputError, band_for, build_query
from recommend import RELAXABLE, broken_rules, load_plans, relax

plans = load_plans()
customers = pd.read_csv(SYNTHETIC_PATH, encoding="utf-8-sig")

print(f"요금제 {len(plans):,}개 · 합성 고객 {len(customers):,}명")
print("컬럼:", list(customers.columns))

## 1. 입력 12개 ↔ 합성 컬럼 매핑

`build_query()`가 받는 값 하나하나가 어디서 오는지, 그리고 **얼마나 채워져 있는지**.
커버리지가 낮으면 그 축으로는 군집도 못 만들고 시나리오도 못 뽑는다.

In [ ]:
MAPPING = [
    ("budget",          "budget_krw",           "요금제 값에서 역산"),
    ("data_band",       "data_gb_month",        "제공량 × 0.9~1.0"),
    ("data_band(무제한)", "data_unlimited_need",  "요금제 플래그"),
    ("voice_unlimited", "voice_unlimited_need",  "요금제 플래그"),
    ("sms_unlimited",   "sms_unlimited_need",    "요금제 플래그"),
    ("voice_minutes",   "voice_minutes_need",    "제공 분수 × 0.9~1.0 (무제한은 없음)"),
    ("sms_count",       "sms_count_need",        "제공 건수 × 0.9~1.0 (무제한은 없음)"),
    ("mvno_ok",         "mvno_ok",               "★ 지어낸 값 (선호도)"),
    ("current_carrier", "current_carrier",       "host_mno (MVNO는 '알뜰폰')"),
    ("age",             "age",                   "KISDI 미디어패널"),
    ("ott_want",        "ott_want",              "요금제가 주는 OTT 중 하나"),
    ("ott_required",    "ott_required",          "★ 지어낸 값 (선호도)"),
    ("current_fee",     "current_fee_krw",       "요금제 값 그대로"),
]

rows = []
for field, col, origin in MAPPING:
    s = customers[col]
    filled = s.notna() & s.ne("") if s.dtype == object else s.notna()
    rows.append({
        "입력": field, "합성 컬럼": col,
        "채워진 비율": f"{filled.mean():.1%}",
        "고유값": s.nunique(),
        "출처": origin,
    })
display(pd.DataFrame(rows))

print("★ 표시는 요금제 스펙에서 역산할 수 없어 확률로 지어낸 축이다.")
print("  분포만 맞춰 뒀을 뿐 실제 선호를 뜻하지 않는다 - 이 축으로 결론을 내면 안 된다.")

## 2. 합성 고객 한 명 → 폼 입력

`persona_to_query()`가 고객 한 줄을 `build_query()` 인자로 바꾼다.
**폼이 거부하는 입력이 나오면 그대로 드러난다** — 조용히 고쳐 넣지 않는다.

In [ ]:
def persona_to_query(row: pd.Series) -> dict:
    """합성 고객 한 명을 폼 입력으로. 폼이 거부하면 InputError가 그대로 올라온다."""
    band = "무제한" if row["data_unlimited_need"] else band_for(row["data_gb_month"])
    want = row["ott_want"]
    return build_query(
        budget=int(row["budget_krw"]),
        data_band=band,
        voice_unlimited=bool(row["voice_unlimited_need"]),
        sms_unlimited=bool(row["sms_unlimited_need"]),
        voice_minutes=None if pd.isna(row["voice_minutes_need"]) else row["voice_minutes_need"],
        sms_count=None if pd.isna(row["sms_count_need"]) else row["sms_count_need"],
        mvno_ok=bool(row["mvno_ok"]),
        current_carrier=row["current_carrier"],
        age=int(row["age"]),
        ott_want=(want,) if isinstance(want, str) and want else (),
        ott_required=bool(row["ott_required"]),
        current_fee=int(row["current_fee_krw"]),
    )


who = customers.iloc[2]
print(who.to_string(), "\n")
print("->", persona_to_query(who))

## 3. 표본 전수 왕복

N명을 통째로 돌려서 세 가지를 본다.

1. **폼이 거부한 고객** — 합성이 만든 값을 우리 폼이 못 받으면 그건 합성 쪽 문제다
2. **규칙 위반** — 예산·데이터·통신사·나이·통화·문자. 하나라도 나오면 `recommend()`가 깨진 것
3. **0건과 완화** — 조건을 얼마나 자주 풀어야 하는지

In [ ]:
N = 500          # 늘리면 정확해지고 느려진다. 4만 전부는 몇 분 걸린다.

sample = customers.sample(N, random_state=20260812)
rejected, results = [], []

for _, row in sample.iterrows():
    try:
        q = persona_to_query(row)
    except InputError as e:
        rejected.append((row["customer_id"], str(e)))
        continue

    got, dropped = relax(plans, q)
    # 조건을 푼 뒤라면 푼 조건은 검사하지 않는다. 원래 요청 기준으로 재면
    # 일부러 푼 것까지 위반으로 잡힌다.
    checked = dict(q)
    checked.update({k: off for k, off, label in RELAXABLE if label in dropped})

    results.append({
        "customer_id": row["customer_id"],
        "n": len(got),
        "위반": ",".join(broken_rules(got, checked)),
        "푼 조건": ",".join(dropped),
        "절감액": int(got["savings"].iloc[0]) if len(got) and "savings" in got else None,
    })

res = pd.DataFrame(results)
print(f"표본 {N}명 중")
print(f"  폼이 거부   : {len(rejected)}명")
print(f"  추천 0건    : {(res['n'] == 0).sum()}명 ({(res['n'] == 0).mean():.1%})")
print(f"  조건 완화   : {res['푼 조건'].ne('').sum()}명 ({res['푼 조건'].ne('').mean():.1%})")
print(f"  규칙 위반   : {res['위반'].ne('').sum()}명   <- 0이어야 한다")

assert res["위반"].eq("").all(), res[res["위반"].ne("")].head()

if rejected:
    print("\n폼이 거부한 사유 (합성 쪽에서 고칠 것)")
    display(pd.Series([msg for _, msg in rejected]).value_counts())

In [ ]:
# 무엇을 얼마나 풀었나. 앞에 있는 것일수록 약한 조건이라 먼저 풀린다.
loosened = res.loc[res["푼 조건"].ne(""), "푼 조건"].str.split(",").explode()
display(loosened.value_counts().rename("푼 횟수").to_frame())

got_any = res[res["n"] > 0]
print(f"추천을 받은 {len(got_any)}명의 1위 절감액")
display(got_any["절감액"].describe().round(0))

## 4. 어떤 입력이 실제로 후보를 자르는가

축을 하나씩 켜 보면서 후보가 얼마나 줄어드는지 본다.
**거의 안 줄어드는 조건은 물어볼 값어치가 없다.**

In [ ]:
from recommend import recommend

base = dict(budget=30_000, data_gb=20.0, data_unlimited=False)
n0 = len(recommend(plans, **base, top_n=99_999))

trials = [
    ("통화 무제한",        dict(voice_unlimited=True)),
    ("문자 무제한",        dict(sms_unlimited=True)),
    ("통화 300분 이상",    dict(voice_minutes=300)),
    ("문자 100건 이상",    dict(sms_count=100)),
    ("3사만",             dict(mvno_ok=False)),
    ("지금 SKT",          dict(current_carrier="SKT")),
    ("만 65세",           dict(age=65)),
    ("넷플릭스 필수",      dict(ott_want=("넷플릭스",), ott_required=True)),
]

display(pd.DataFrame([
    {"조건": label,
     "남는 후보": (n := len(recommend(plans, **base, **kw, top_n=99_999))),
     "남는 비율": f"{n / n0:.1%}"}
    for label, kw in trials
]).assign(기준=f"조건 없음 {n0}개"))